<h1><center>A full business solution</center></h1>

In [46]:
# import libraries
import os
import sys
sys.path.insert(0, os.path.abspath(".."))  # so Python finds week1/scraper.py
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [47]:
# Initialize and constants
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10 :
    print("API key looks good so far.")
else:
    print("There might be a problem with you API key. Please check it and try again.")

MODEL = "gpt-5-nano"
openai = OpenAI()

API key looks good so far.


In [48]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## 1. Have GPT-5-nano figure out which links are relevant

In [49]:
links_system_prompt = """
You are provided with a list of links from a website. Your
task is to decide which of the links would be most relevant
to include in a brochure about the company such as links to 
an About page, or a Company page, or Careers/Jobs page.
You should respond in JSON as in this example:

{
"Links": [
{"type": "about page", "url": "https://full.url/goes/here/about"},
{"type": "careers page", "url": "https://another.full.url/careers"}
]
}


 """

In [50]:
def get_links_user_prompt(url):
    user_prompt = f""" 
Here is the list of links on the wehsite {url} - Please
decide which of theses are relevant web link for a 
brochure about the company, respond with full https URL
in JSON format. Do not include Terms of Service, Privacy,
email links.

Links (some might be relative links):
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [51]:
print(get_links_user_prompt("https://edwarddonner.com"))

 
Here is the list of links on the wehsite https://edwarddonner.com - Please
decide which of theses are relevant web link for a 
brochure about the company, respond with full https URL
in JSON format. Do not include Terms of Service, Privacy,
email links.

Links (some might be relative links):
#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-

In [52]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system",
             "content": links_system_prompt},
             {"role": "user",
              "content": get_links_user_prompt(url)}
        ],
        response_format = {"type": "json_object"}   
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [53]:
select_relevant_links("https://edwarddonner.com")

{'Links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'}]}

In [54]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system",
             "content": links_system_prompt},
             {"role": "user",
              "content": get_links_user_prompt(url)}
        ],
        response_format = {"type": "json_object"}   
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['Links'])} relevant links")
    return links

In [55]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'Links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'brand/company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [56]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'Links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## 2. Make the brochure

In [57]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    # Find the list of links regardless of exact key name/casing
    links = relevant_links.get("Links") or relevant_links.get("links") or []
    for link in links:
        result += f"\n\n### link: {link['type']}\n"
        result += fetch_website_contents(link['url'])
    return result

In [58]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling
Updated
1 day ago
•
16.4k
•
1.34k
prism-ml/Ternary-Bonsai-27B-gguf
Updated
4 days ago
•
432k
•
895
baidu/Unlimited-OCR
Updated
about 10 hours ago
•
2.24M
•
2.59k
prism-ml/Bonsai-27B-gguf
Updated
4 days ago
•
1.4M
•
569
zai-org/GLM-5.2
Updated
20 days ago
•
545k

In [59]:
brochure_system_prompt = """ 
You are an assistant that analyzes the contents of several 
relevant pages from a company website and creates a short
brochure about the company for prospective customers, 
investors, and recruits. Respond in markdown without code 
blocks. Include the details of company culture, customers,
and careers/jobs if you have the information.
"""


In [67]:
def get_brochure_user_prompt(company_name, url):
    user_prompt =f""" 
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant
pages; use this information to build a short brochure of the
company in markdown without code blocks.
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters.
    return user_prompt


In [69]:
print(get_brochure_user_prompt("Hugging Face", "https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links
 
You are looking at a company called: Hugging Face
Here are the contents of its landing page and other relevant
pages; use this information to build a short brochure of the
company in markdown without code blocks.
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling
Updated
1 day ago
•
16.4k
•
1.35k
prism-ml/Ter

In [70]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system",
             "content": brochure_system_prompt},
             {"role": "user",
              "content": get_brochure_user_prompt(company_name, url)}
              ], 
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [71]:
create_brochure("Hugging Face", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


# Hugging Face: The AI community building the future

The platform where the machine learning community collaborates on models, datasets, and applications. The Home of Machine Learning.

---

## About Hugging Face

- The AI community building the future. A collaboration platform where researchers, engineers, and end users learn, share, and build open-source ML together.
- The Hub is a central place to share, explore, discover, and experiment with open-source ML.
- Brand mission: empower the next generation of ML engineers, scientists, and users to learn, collaborate, and build an open and ethical AI future together.
- Core idea: open collaboration accelerates innovation in machine learning and AI.

---

## What we offer (for customers, partners, and enterprises)

- Core platform
  - Models, Datasets, Spaces, Buckets
  - HuggingChat, Inference Endpoints, Inference Providers
  - Team & Enterprise plans, Hugging Face PRO, Enterprise Support
- Enterprise-ready capabilities
  - Inference Providers, Inference Endpoints, Storage Buckets
- Community-driven AI
  - Explore and run 2M+ models, browse 500k+ datasets, and experiment with 1M+ Spaces applications
- Accessibility and speed
  - A fast, collaborative environment to host and work on public models, datasets, and applications
- Brand and ecosystem
  - Brand assets and resources for teams building with Hugging Face
  - Documentation, Learn resources, and community forums to accelerate adoption

Note: The platform highlights a broad, open ecosystem designed to move faster together, with a strong emphasis on collaboration and open-source ML.

---

## For investors

- A vibrant open-source ML ecosystem underpinning rapid innovation and community-driven development.
- Large-scale, globally adopted platform with 2M+ models and 500k+ datasets available to the community, plus Spaces with 1M+ applications.
- A clear path from research to production via enterprise offerings: PRO, Enterprise Support, Inference Endpoints, and Storage Buckets.
- A strategic position at the heart of the AI revolution, powered by a growing community and a fast-moving product roadmap.

---

## For talent and careers

- Culture and mission
  - A mission-driven, collaborative culture focused on open, ethical AI and empowering the ML community.
  - The brand emphasizes learning, collaboration, and sharing work to advance the field.
- What you’ll find
  - Opportunities to contribute to open-source ML libraries, collaborate with researchers and engineers, and help scale a global AI ecosystem.
  - Roles across engineering, research, product, operations, and community teams (explore the Careers page for current openings).
- How to get involved
  - Visit the Hugging Face Careers page for current opportunities and to learn more about joining the team.
  - Engage with a global community through forums, Discord, GitHub, and the brand ecosystem.

---

## Brand, assets, and design notes

- Brand colors (illustrative for partnerships and materials): HF Yellow (#FFD21E), HF Orange (#FF9D00), and a neutral slate (#6B7280).
- Brand assets include official logos and guidelines to maintain consistency across materials.
- Core brand statement: “The AI community building the future.” The Hugging Face Hub is a central place for sharing and exploring open-source ML, enabling the next generation of ML engineers, scientists, and end users to learn, collaborate, and share their work to build an open and ethical AI future together.

---

If you’d like, I can tailor this brochure to a specific audience (customers, investors, or recruiters) and adjust the tone or add call-to-action sections with links to the brand and careers pages.

## 3. A minor improvement

In [72]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = [
            {"role": "system",
             "content": brochure_system_prompt},
            {"role": "user",
             "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream = True
    )
    response = ""
    display_handle = display(Markdown(""), 
                             display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
        

In [73]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the premier collaboration platform for the global machine learning (ML) community. It empowers researchers, developers, and enterprises to create, discover, and share ML models, datasets, and applications at scale. Hugging Face hosts over **2 million models**, **500k datasets**, and **1 million+ applications**, serving as a vibrant ecosystem where innovation and collaboration accelerate the development of AI solutions.

The platform offers tools such as:

- **Models:** Access to a vast library of pre-trained and fine-tuned models for various AI tasks.
- **Datasets:** Extensive and continually expanding datasets curated for diverse ML applications.
- **Spaces:** Interactive machine learning apps hosted directly on the platform.
- **Buckets:** Scalable storage solutions for datasets and models.
- **Enterprise Solutions:** Including Hugging Face PRO, enterprise support, inference providers, and endpoints.

---

## Company Culture & Community

Hugging Face fosters an open, inclusive, and collaborative culture that values transparency, creativity, and community-driven development. The company thrives on empowering contributors worldwide through:

- **Active Community Engagement:** Via Discord, forums, GitHub, and blog platforms where ideas and innovations are shared.
- **Educational Resources:** Detailed documentation, tutorials, and daily research paper posts to support learning.
- **Open Collaboration:** Unlimited hosting and collaboration capabilities on public ML assets encourage open science and reproducibility.
- **Innovation Driven:** Hugging Face regularly surfaces trending models and applications, spotlighting the latest breakthroughs and collaborations in AI.

---

## Customers & Applications

Hugging Face serves a diverse customer base, ranging from individual researchers and developers to large enterprises. Its platform supports various AI applications, including but not limited to:

- Natural Language Processing (NLP)
- Computer Vision
- Speech Recognition and Voice Processing
- Data Science and Reasoning Agents
- Custom enterprise AI solutions

Customers leverage Hugging Face to accelerate their AI projects, integrate powerful ML inference into their products, and benefit from scalable, reliable AI infrastructure solutions.

---

## Careers at Hugging Face

Joining Hugging Face means becoming part of a mission-driven company aiming to democratize AI. The company looks for passionate individuals who are excited about:

- Building and scaling cutting-edge machine learning infrastructures
- Engaging deeply with an active AI and research community
- Driving open-source and open-science initiatives forward
- Collaborating in an agile, innovative, and inclusive environment

Hugging Face offers opportunities across engineering, research, product management, and customer success teams, emphasizing flexibility, continuous learning, and impact.

---

## Why Choose Hugging Face?

- **Largest Collection of ML Resources:** Explore and contribute to millions of models and datasets.
- **Community-First Platform:** Connect and collaborate with thousands of AI practitioners globally.
- **Enterprise Ready:** Reliable and scalable AI infrastructure tailored to business needs.
- **Innovative Tools & Apps:** Use and develop interactive AI applications to solve real-world problems.
- **Open Source & Transparent:** A commitment to open science fuels trust and progress.

---

**Discover the future of AI—join the Hugging Face community today!**

Explore more at: [huggingface.co](https://huggingface.co)  
Connect on Discord, GitHub, and forums to start collaborating!

In [74]:
stream_brochure("Real Python", "https://realpython.com")

Selecting relevant links for https://realpython.com by calling gpt-5-nano
Found 13 relevant links


# Real Python: Your Premier Python Learning Platform

---

## About Real Python

Real Python is a leading provider of online Python education and one of the largest language-specific online communities, serving over 1 million developers, data scientists, and machine learning engineers every month. We empower Python enthusiasts at all levels through high-quality, curated learning resources including tutorials, books, courses, and live classes.

### What Makes Real Python Special?

Think of Real Python as a “gym for Pythonistas,” where you can grow your Python skills deeply and steadily over time—without wasting hours hunting down unreliable tutorials or struggling through poorly structured courses.

- **Beginners** learn fundamentals through engaging, hands-on projects.
- **Intermediate developers** face real-world challenges and best practices.
- **Advanced practitioners** dive deep into performance optimization and advanced techniques.

Our approach emphasizes **learn-by-doing**, combined with up-to-date, trustworthy materials to help you become a confident and efficient Python programmer.

---

## Our Offerings

### Personalized Learning Paths  
Tailor your own learning plan based on your goals and skill level, accelerating your growth with guided study routes.

### In-depth Tutorials & Video Courses  
Explore step-by-step tutorials and comprehensive video courses on a wide range of Python topics.

### Practice & Assessment  
Reinforce your skills with quizzes and coding exercises to track your progress effectively.

### Get Help  
- **Code Mentor (Beta):** Receive personalized coding assistance and learning tools.  
- **Office Hours:** Join live Q&A calls with Python experts to clarify doubts.  
- **Community Chat:** Connect and learn alongside other Pythonistas.

### Deeper Learning  
- **Live Courses:** Participate in instructor-led live Python classes.  
- **Books:** Complement your learning offline.  
- **Podcast:** Stay updated with the latest insights in the Python ecosystem.

---

## Our Community & Customers

Our thriving community includes developers at all experience levels, data scientists, and machine learning engineers worldwide who rely on Real Python for professional growth and skill mastery. Members appreciate our curated and reliable content that saves time and drives meaningful learning progress.

---

## Careers at Real Python

We seek passionate Pythonistas eager to contribute to the growth of the Python community and help others learn. Opportunities include content creation, mentoring, software development, and community management. If you are excited about Python education and want to be part of a vibrant team, Real Python is the place to grow your career.

*Visit the Real Python website to view current job openings and learn how to become a contributor.*

---

## Join Real Python Today

Unlock unlimited access to all Real Python resources and become a Python expert on your terms. Whether you are just starting out or looking to deepen advanced skills, Real Python offers the tools and community to get there.

---

*For more information, visit:* [realpython.com](https://realpython.com)

---

*Real Python — Learn Python. Code Confidently.*